# 1. 목적과 범위

주거실태조사와 한국부동산원 월별 자료로 2016–2024년 주거지표 3종을 전국과 17개 시도에 대해 산출하고, 현재 주택가격과 기존 대체지표를 비교한다.

- 전체 가구 자가점유비율: 점유형태가 유효하고 조사 가중치가 양수인 전체 가구 중 자가(1) 가구의 조사 가중비율
- 임차가구 연간 주거비 HCC: 임차가구(점유형태 2–6)의 `보증금 × 적용 전월세전환율 / 100 + 월세 × 12`를 지역별로 비가중 산술평균한 값
- 현재 주택가격: 0717 원자료의 현재 주택가격(만원) 유효 응답을 제주 원 보고서의 `1/n` 정의에 따라 지역별로 비가중 산술평균한 값

자가점유비율과 HCC는 0727, 현재 주택가격은 0717 원자료만 사용하며 두 자료를 행 단위로 병합하지 않는다. 지표별 가중치 정책은 서로 다르다. 자가점유비율만 조사 가중치를 사용하고, HCC와 현재 주택가격은 가구별 유효값의 산술평균이므로 조사 가중치를 사용하지 않는다. 2016년 충남·세종 결합표본의 세종 결과는 임의로 채우지 않는다.

## 2. 최종 지표 정의와 가중치 정책

전체 가구 자가점유비율은 점유형태 1–7과 양수 조사 가중치를 가진 전체 가구를 분모로 하고, 그중 자가(1) 가구의 가중치 합을 분자로 계산한다. 출생연도·가구주 관계·연령 조건은 사용하지 않는다.

임차가구 연간 주거비 HCC는 점유형태 2–6을 대상으로 하며 무상거주(7)는 제외한다. 전세(2)의 월세 공란과 보증금 없는 임차(4–6)의 보증금 공란만 구조적 0으로 처리하고, 필수 금액의 일반 결측·`9999999`·음수·비정상값은 제외한다. 지역별 HCC는 조사 가중치 없이 산술평균한다. 본계열은 KOSIS 표시 정밀도 기반 재현 경로를 사용하고 월별 원 정밀도 평균 결과는 QA 민감도 계열로만 보존한다. 원 보고서는 월별 값의 연평균 사용을 명시하지만 중간 정밀도 규칙은 제시하지 않았다는 한계가 있다.

현재 주택가격은 0717 원자료의 유효 현재 주택가격을 제주 원 보고서의 `1/n` 산술평균 정의로 집계한다. 조사 가중치를 사용하지 않으며 기존 산식과 지역별 결과를 유지한다.

## 3. 라이브러리와 원자료 경로 설정

저장소 루트를 기준으로 0717 현재 주택가격 원자료, 0727 주거실태조사 원자료, 공통 보조자료와 `data/processed` 경로를 분리해 설정한다. 산출 디렉터리가 없으면 생성하며, 같은 산출물 경로의 기존 파일을 회귀검증 기준으로 사용하지 않는다.

In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

np.random.seed(42)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / "data").is_dir() else cwd.parent
RAW_ROOT = REPO_ROOT / "data" / "raw" / "구조환경지수 원데이터 구축용" / "주거실태조사"
PRICE_RAW_DIR = RAW_ROOT / "0717"
HOUSING_RAW_DIR = RAW_ROOT / "0727"
PROCESSED_DIR = REPO_ROOT / "data" / "processed"
YEARS = list(range(2016, 2025))
INVALID_CODE = 9_999_999

assert PRICE_RAW_DIR.is_dir(), PRICE_RAW_DIR
assert HOUSING_RAW_DIR.is_dir(), HOUSING_RAW_DIR
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
print(f"저장소: {REPO_ROOT}")
print(f"주택가격 원자료: {PRICE_RAW_DIR}")
print(f"주거실태조사 원자료: {HOUSING_RAW_DIR}")

저장소: D:\University\yumocha\yumocha-issue49
주택가격 원자료: D:\University\yumocha\yumocha-issue49\data\raw\구조환경지수 원데이터 구축용\주거실태조사\0717
주거실태조사 원자료: D:\University\yumocha\yumocha-issue49\data\raw\구조환경지수 원데이터 구축용\주거실태조사\0727


## 4. 0717·0727 연도별 원자료 매핑

각 폴더에서 2016–2024년 파일이 연도별 정확히 하나인지 검증한다. 0727은 전체 가구 자가점유비율과 임차가구 HCC에, 0717은 현재 주택가격에만 사용하며 서로 병합하지 않는다. 최종 지표에 불필요한 출생연도·가구주 관계·소득 열은 입력 검증과 계산에서 사용하지 않는다.

In [2]:
COL_TENURE = "문7. 귀 댁의 점유형태는 어디에 해당됩니까?"
COL_DEPOSIT = "문15. 현재 살고 계신 주택의 임차료는 얼마입니까?_보증금(만원)"
COL_RENT = "문15. 현재 살고 계신 주택의 임차료는 얼마입니까?_월세(만원)"
WEIGHT_COLUMN_BY_YEAR = {2016: "모집단가중치", 2020: "최종 가중치"}

housing_files = {}
header_rows = []
for year in YEARS:
    matches = sorted(HOUSING_RAW_DIR.glob(f"{year}_일반가구_*.csv"))
    assert len(matches) == 1, f"{year}년 파일 후보 {len(matches)}개: {matches}"
    path = matches[0]
    columns = pd.read_csv(path, encoding="cp949", nrows=0).columns.tolist()
    assert len(columns) == len(set(columns)), f"{year}년 중복 헤더"
    assert all(str(column).strip() for column in columns), f"{year}년 빈 헤더"
    weight_column = WEIGHT_COLUMN_BY_YEAR.get(year, "최종가중치")
    required_columns = ["시도", COL_TENURE, COL_DEPOSIT, COL_RENT, weight_column]
    missing_columns = [column for column in required_columns if column not in columns]
    assert not missing_columns, f"{year}년 필수 헤더 누락: {missing_columns}"
    housing_files[year] = path
    header_rows.append(
        {
            "연도": year,
            "파일명": path.name,
            "전체 열 수": len(columns),
            "검증한 필수 열 수": len(required_columns),
            "자가점유비율 가중치 헤더": weight_column,
        }
    )

header_check = pd.DataFrame(header_rows)
print(header_check.to_string(index=False))

  연도                          파일명  전체 열 수  검증한 필수 열 수 자가점유비율 가중치 헤더
2016 2016_일반가구_20260727_16717.csv       7           5        모집단가중치
2017 2017_일반가구_20260727_16717.csv      28           5         최종가중치
2018 2018_일반가구_20260727_16717.csv      28           5         최종가중치
2019 2019_일반가구_20260727_16717.csv      28           5         최종가중치
2020 2020_일반가구_20260727_16717.csv      28           5        최종 가중치
2021 2021_일반가구_20260727_94806.csv      28           5         최종가중치
2022 2022_일반가구_20260727_94806.csv      28           5         최종가중치
2023 2023_일반가구_20260727_94806.csv      28           5         최종가중치
2024 2024_일반가구_20260727_94806.csv      28           5         최종가중치


### 0717 주택가격 원자료 검증

0717은 현재 주택가격 산출에만 사용하고 0727과 행 단위로 병합하지 않는다. 코드북은 제공되지 않았으며, 헤더의 `현재 주택가격(만원)` 단위와 사용자 확인 기준을 적용한다.

In [3]:
COL_PRICE = "문12. 현재 살고 계신 주택의 가격은 얼마입니까? - 현재 주택가격(만원)"
PRICE_EXPECTED_COLUMNS = {2016: 7, 2017: 8, **{year: 6 for year in range(2018, 2025)}}
price_files = {}
price_file_rows = []
for year in YEARS:
    matches = sorted(PRICE_RAW_DIR.glob(f"{year}_일반가구_*.csv"))
    assert len(matches) == 1, f"0717 {year}년 파일 후보 {len(matches)}개: {matches}"
    path = matches[0]
    columns = pd.read_csv(path, encoding="cp949", nrows=0).columns.tolist()
    assert len(columns) == PRICE_EXPECTED_COLUMNS[year], f"0717 {year}년 열 수 불일치"
    assert len(columns) == len(set(columns)) and all(str(column).strip() for column in columns)
    assert all(column in columns for column in ["시도", COL_TENURE, COL_PRICE])
    price_files[year] = path
    price_file_rows.append(
        {"연도": year, "파일명": path.name, "열 수": len(columns), "가격 변수": COL_PRICE}
    )

assert set(price_files.values()).isdisjoint(housing_files.values()), "0717·0727 파일이 중복 선택됨"
price_file_check = pd.DataFrame(price_file_rows)
print("0717·0727 모두 연도별 1개 파일 선택, 두 자료 경로 분리 확인")
print(price_file_check.to_string(index=False))

0717·0727 모두 연도별 1개 파일 선택, 두 자료 경로 분리 확인
  연도                          파일명  열 수                                      가격 변수
2016 2016_일반가구_20260717_46033.csv    7 문12. 현재 살고 계신 주택의 가격은 얼마입니까? - 현재 주택가격(만원)
2017 2017_일반가구_20260717_46033.csv    8 문12. 현재 살고 계신 주택의 가격은 얼마입니까? - 현재 주택가격(만원)
2018 2018_일반가구_20260717_46033.csv    6 문12. 현재 살고 계신 주택의 가격은 얼마입니까? - 현재 주택가격(만원)
2019 2019_일반가구_20260717_46033.csv    6 문12. 현재 살고 계신 주택의 가격은 얼마입니까? - 현재 주택가격(만원)
2020 2020_일반가구_20260717_46033.csv    6 문12. 현재 살고 계신 주택의 가격은 얼마입니까? - 현재 주택가격(만원)
2021 2021_일반가구_20260717_12250.csv    6 문12. 현재 살고 계신 주택의 가격은 얼마입니까? - 현재 주택가격(만원)
2022 2022_일반가구_20260717_12250.csv    6 문12. 현재 살고 계신 주택의 가격은 얼마입니까? - 현재 주택가격(만원)
2023 2023_일반가구_20260717_12250.csv    6 문12. 현재 살고 계신 주택의 가격은 얼마입니까? - 현재 주택가격(만원)
2024 2024_일반가구_20260717_12250.csv    6 문12. 현재 살고 계신 주택의 가격은 얼마입니까? - 현재 주택가격(만원)


## 5. KOSIS 전월세전환율 전처리와 연평균 계산

KOSIS CSV의 `종합`·`지역별 전월세전환율`만 선택한다. 월별 값 `5.8`은 5.8%이므로 HCC 계산에서 `/100`을 정확히 한 번 적용한다. 내보낸 파일의 `단위` 열이 공란이면 이를 기록하되, 과업에서 확정한 퍼센트 의미를 적용한다.

In [4]:
kosis_matches = sorted(RAW_ROOT.glob("*KOSIS*전월세전환율*.csv"))
assert len(kosis_matches) == 1, f"KOSIS 파일 후보 {len(kosis_matches)}개: {kosis_matches}"
KOSIS_PATH = kosis_matches[0]
kosis_raw = pd.read_csv(KOSIS_PATH, encoding="cp949")
trailing_columns = [column for column in kosis_raw.columns if str(column).startswith("Unnamed:")]
assert all(kosis_raw[column].isna().all() for column in trailing_columns), (
    "비어 있지 않은 Unnamed 열 존재"
)
kosis = kosis_raw.drop(columns=trailing_columns)
id_columns = ["주택유형별", "지역별", "항목", "단위"]
assert all(column in kosis.columns for column in id_columns)
month_pattern = re.compile(r"^(20(?:16|17|18|19|20|21|22|23|24))\.(0[1-9]|1[0-2]) 월$")
month_columns = [column for column in kosis.columns if month_pattern.fullmatch(str(column))]
assert len(month_columns) == 9 * 12, len(month_columns)
unit_values = {str(value).strip() for value in kosis["단위"].dropna() if str(value).strip()}
assert not unit_values or unit_values == {"%"}, f"예상 밖 단위: {unit_values}"
unit_note = (
    "KOSIS 단위 열은 공란; 과업 정의에 따라 값을 퍼센트로 해석"
    if not unit_values
    else "KOSIS 단위: %"
)
print(f"KOSIS 파일: {KOSIS_PATH.name}")
print(unit_note)

KOSIS 파일: 16-24 KOSIS 지역별 전월세전환율.csv
KOSIS 단위 열은 공란; 과업 정의에 따라 값을 퍼센트로 해석


### 지역명 정규화와 연평균 전환율

전국과 17개 시도의 2016–2024년 각 연도에 월별 관측치가 정확히 12개인지 먼저 검증한 뒤 단순평균한다. `강원특별자치도`, `전북특별자치도` 표기는 각각 `강원`, `전북`으로 정규화한다.

In [5]:
REGION_ORDER = [
    "서울",
    "부산",
    "대구",
    "인천",
    "광주",
    "대전",
    "울산",
    "세종",
    "경기",
    "강원",
    "충북",
    "충남",
    "전북",
    "전남",
    "경북",
    "경남",
    "제주",
]
REGION_NORMALIZE = {"강원특별자치도": "강원", "전북특별자치도": "전북", "제주특별자치도": "제주"}
target_regions = ["전국", *REGION_ORDER]
selected = kosis.loc[
    (kosis["주택유형별"] == "종합") & (kosis["항목"] == "지역별 전월세전환율")
].copy()
selected["지역"] = selected["지역별"].replace(REGION_NORMALIZE)
selected = selected.loc[selected["지역"].isin(target_regions)]
assert selected["지역"].value_counts().eq(1).all(), "대상 지역 중복"
assert set(selected["지역"]) == set(target_regions), "전국 또는 17개 시도 누락"
conversion_long = selected.melt(
    id_vars=["지역"], value_vars=month_columns, var_name="기준연월", value_name="전월세전환율"
)
date_parts = conversion_long["기준연월"].str.extract(month_pattern)
conversion_long["연도"] = date_parts[0].astype(int)
conversion_long["월"] = date_parts[1].astype(int)
conversion_long["전월세전환율"] = pd.to_numeric(conversion_long["전월세전환율"], errors="coerce")
month_counts = conversion_long.groupby(["지역", "연도"], observed=True)["월"].nunique()
nonmissing_counts = conversion_long.groupby(["지역", "연도"], observed=True)["전월세전환율"].count()
assert month_counts.eq(12).all() and nonmissing_counts.eq(12).all(), (
    "지역-연도별 12개월 완전성 실패"
)
annual_conversion = conversion_long.groupby(["지역", "연도"], observed=True)["전월세전환율"].mean()
assert annual_conversion.size == 18 * 9
print(f"KOSIS 12개월 완전성: {int(month_counts.eq(12).sum())}/{month_counts.size} 지역-연도 통과")
print(annual_conversion.unstack("연도").round(3).to_string())

KOSIS 12개월 완전성: 162/162 지역-연도 통과
연도   2016   2017   2018   2019   2020   2021   2022   2023   2024
지역                                                               
강원  8.185  7.618  7.270  7.060  6.746  6.837  6.626  6.919  6.919
경기  6.628  6.393  6.372  6.266  5.953  5.901  6.088  6.408  6.291
경남  8.063  7.661  7.291  7.157  6.881  6.990  6.855  6.673  6.468
경북  9.890  9.486  9.079  8.861  8.515  8.404  7.884  7.440  7.632
광주  7.515  7.084  6.956  6.856  6.461  6.273  6.197  6.022  5.896
대구  7.759  7.421  7.326  7.314  7.119  6.830  6.469  6.207  6.070
대전  7.489  7.333  7.117  6.891  6.556  6.106  5.983  6.275  6.522
부산  7.323  7.039  6.907  6.541  6.316  5.982  5.919  6.013  6.094
서울  5.888  5.497  5.347  5.189  4.929  4.740  4.812  5.166  5.113
세종  5.684  5.158  5.456  5.339  5.111  5.065  5.528  6.138  5.965
울산  7.614  7.391  7.201  7.155  6.809  6.652  6.621  6.822  6.746
인천  7.094  6.821  6.661  6.453  6.035  5.874  5.925  6.548  6.307
전국  6.720  6.382  6.254  6.120  5.814  5.67

## 6. 지역 매핑과 구조적 결측 규칙

전국은 원자료 전체에서 직접 산출하고 시도는 공통 지역 순서로 정렬한다. 2016년 시도코드 29·34는 기존 충남·세종 결합표본 규칙에 따라 충남으로 집계하고 세종은 세 지표 모두 구조적 결측으로 유지한다.

In [6]:
SURVEY_REGION_MAP = {
    11: "서울",
    21: "부산",
    22: "대구",
    23: "인천",
    24: "광주",
    25: "대전",
    26: "울산",
    29: "세종",
    31: "경기",
    32: "강원",
    33: "충북",
    34: "충남",
    35: "전북",
    36: "전남",
    37: "경북",
    38: "경남",
    39: "제주",
}
OUTPUT_REGION_ORDER = ["전국", *REGION_ORDER]
EXPECTED_STRUCTURAL_MISSING = {(2016, "세종")}

assert list(SURVEY_REGION_MAP.values()) == REGION_ORDER
assert len(OUTPUT_REGION_ORDER) == 18 and len(set(OUTPUT_REGION_ORDER)) == 18
print(f"최종 지역 순서: {len(OUTPUT_REGION_ORDER)}개(전국 + 17개 시도)")

최종 지역 순서: 18개(전국 + 17개 시도)


## 7. 지표별 집계·표시 정책

| 지표 | 대상 | 지역 집계 | 본계열 표시 |
|---|---|---|---|
| 전체 가구 자가점유비율 | 점유형태 1–7 및 양수 가중치 가구 | 조사 가중비율 | 소수 첫째 자리 |
| 임차가구 연간 주거비 HCC | 점유형태 2–6 중 유효 금액 가구 | 비가중 산술평균 | 정수 만원 |
| 현재 주택가격 | 유효 현재 주택가격 응답 | 비가중 `1/n` 산술평균 | 소수 첫째 자리(정수 보조 표시) |

각 결과표는 원 계산값을 별도 변수와 QA 표에 보존하고 CSV에는 표시용 반올림값을 기록한다. HCC 원 정밀도 민감도는 QA에만 두며 본계열 결과표와 CSV에 혼합하지 않는다.

In [7]:
indicator_specs = pd.DataFrame(
    [
        {"세부지표": "전체 가구 자가점유비율", "자료": "0727", "집계": "조사 가중비율", "표시": "소수 첫째 자리"},
        {"세부지표": "임차가구 연간 주거비 HCC", "자료": "0727", "집계": "비가중 산술평균", "표시": "정수 만원"},
        {"세부지표": "현재 주택가격", "자료": "0717", "집계": "비가중 1/n 산술평균", "표시": "소수 첫째 자리"},
    ]
)
assert indicator_specs["세부지표"].nunique() == 3
indicator_specs

,세부지표,자료,집계,표시
0,전체 가구 자가점유비율,0727,조사 가중비율,소수 첫째 자리
1,임차가구 연간 주거비 HCC,0727,비가중 산술평균,정수 만원
2,현재 주택가격,0717,비가중 1/n 산술평균,소수 첫째 자리


In [8]:
OWN_REGION_ORDER = OUTPUT_REGION_ORDER
OWN_EXPECTED_MISSING = EXPECTED_STRUCTURAL_MISSING


def load_ownership_year(year, path):
    weight_source = WEIGHT_COLUMN_BY_YEAR.get(year, "최종가중치")
    frame = pd.read_csv(
        path, encoding="cp949", usecols=["시도", COL_TENURE, weight_source]
    ).rename(
        columns={
            "시도": "시도코드",
            COL_TENURE: "점유형태",
            weight_source: "가구가중치",
        }
    )
    frame["시도코드"] = pd.to_numeric(frame["시도코드"], errors="coerce")
    frame["지역"] = frame["시도코드"].map(SURVEY_REGION_MAP)
    if year == 2016:
        frame.loc[frame["시도코드"].isin([29, 34]), "지역"] = "충남"
    assert frame["지역"].notna().all(), f"{year}년 미매핑 시도코드 존재"
    frame["점유형태"] = pd.to_numeric(frame["점유형태"], errors="coerce")
    frame["가구가중치"] = pd.to_numeric(frame["가구가중치"], errors="coerce")
    return frame


def calculate_ownership_year(year, path):
    frame = load_ownership_year(year, path)
    valid_tenure = frame["점유형태"].isin(range(1, 8))
    valid_weight = frame["가구가중치"].notna() & frame["가구가중치"].gt(0)
    indicators = {}
    qa_rows = []

    for region in OWN_REGION_ORDER:
        region_mask = (
            pd.Series(True, index=frame.index)
            if region == "전국"
            else frame["지역"].eq(region)
        )
        denominator_mask = region_mask & valid_tenure & valid_weight
        numerator_mask = denominator_mask & frame["점유형태"].eq(1)
        denominator_weight = float(frame.loc[denominator_mask, "가구가중치"].sum())
        numerator_weight = float(frame.loc[numerator_mask, "가구가중치"].sum())
        ratio = (
            numerator_weight / denominator_weight * 100
            if denominator_weight > 0
            else np.nan
        )
        indicators[region] = ratio
        qa_rows.append(
            {
                "연도": year,
                "지역": region,
                "전체 원표본 수": int(region_mask.sum()),
                "점유형태 유효 표본 수": int((region_mask & valid_tenure).sum()),
                "자가 표본 수": int((region_mask & frame["점유형태"].eq(1)).sum()),
                "유효 가중치 표본 수": int((region_mask & valid_weight).sum()),
                "분모 가중치 합": denominator_weight,
                "분자 가중치 합": numerator_weight,
                "자가점유비율": ratio,
                "제외된 점유형태 결측·비정상 코드 수": int(
                    (region_mask & ~valid_tenure).sum()
                ),
                "가중치 결측·0·음수 건수": int((region_mask & ~valid_weight).sum()),
            }
        )

    return pd.Series(indicators, dtype=float), pd.DataFrame(qa_rows)


own_by_year = {}
own_qa_frames = []
for year in YEARS:
    own_by_year[year], year_qa = calculate_ownership_year(year, housing_files[year])
    own_qa_frames.append(year_qa)

own_qa = pd.concat(own_qa_frames, ignore_index=True)
calculated_own_qa = own_qa.loc[own_qa["자가점유비율"].notna()].copy()
observed_missing = set(
    own_qa.loc[own_qa["자가점유비율"].isna(), ["연도", "지역"]].itertuples(
        index=False, name=None
    )
)
result_counts_by_year = calculated_own_qa.groupby("연도")["지역"].nunique()
jeju_own_2023_raw = float(
    calculated_own_qa.loc[
        calculated_own_qa["연도"].eq(2023) & calculated_own_qa["지역"].eq("제주"),
        "자가점유비율",
    ].iloc[0]
)

own_qa_checks = pd.DataFrame(
    [
        {
            "검증 항목": "모든 비율 0–100%",
            "결과": "PASS"
            if calculated_own_qa["자가점유비율"].between(0, 100, inclusive="both").all()
            else "FAIL",
        },
        {
            "검증 항목": "분자 가중치 합 ≤ 분모 가중치 합",
            "결과": "PASS"
            if (
                calculated_own_qa["분자 가중치 합"]
                <= calculated_own_qa["분모 가중치 합"] + 1e-9
            ).all()
            else "FAIL",
        },
        {
            "검증 항목": "모든 산출 지역 분모 > 0",
            "결과": "PASS"
            if calculated_own_qa["분모 가중치 합"].gt(0).all()
            else "FAIL",
        },
        {
            "검증 항목": "필수 연도 2016–2024 누락 없음",
            "결과": "PASS" if set(own_by_year) == set(YEARS) else "FAIL",
        },
        {
            "검증 항목": "전국·17개 시도 결과 누락 확인",
            "결과": "PASS"
            if (
                len(own_qa) == len(YEARS) * len(OWN_REGION_ORDER)
                and observed_missing == OWN_EXPECTED_MISSING
                and result_counts_by_year.loc[2016] == 17
                and result_counts_by_year.drop(index=2016).eq(18).all()
            )
            else "FAIL",
        },
        {
            "검증 항목": "2023년 제주 57.1%",
            "결과": "PASS" if round(jeju_own_2023_raw, 1) == 57.1 else "FAIL",
        },
    ]
)
assert own_qa_checks["결과"].eq("PASS").all(), own_qa_checks
assert np.isclose(jeju_own_2023_raw, 57.0893148, atol=5e-8)
print(own_qa_checks.to_string(index=False))
print(
    f"자가점유비율 결과: {len(calculated_own_qa)}/{len(own_qa)}개 지역·연도, "
    f"구조적 결측: {sorted(observed_missing)}"
)
print(f"2023년 제주 원 계산값: {jeju_own_2023_raw:.10f}%, 반올림: {jeju_own_2023_raw:.1f}%")
own_qa.loc[
    own_qa["연도"].eq(2023) & own_qa["지역"].eq("제주"),
]

                검증 항목   결과
         모든 비율 0–100% PASS
  분자 가중치 합 ≤ 분모 가중치 합 PASS
      모든 산출 지역 분모 > 0 PASS
필수 연도 2016–2024 누락 없음 PASS
   전국·17개 시도 결과 누락 확인 PASS
       2023년 제주 57.1% PASS
자가점유비율 결과: 161/162개 지역·연도, 구조적 결측: [(2016, '세종')]
2023년 제주 원 계산값: 57.0893148298%, 반올림: 57.1%


,연도,지역,전체 원표본 수,점유형태 유효 표본 수,자가 표본 수,유효 가중치 표본 수,분모 가중치 합,분자 가중치 합,자가점유비율,제외된 점유형태 결측·비정상 코드 수,가중치 결측·0·음수 건수
143,2023,제주,1824,1824,1115,1824,276225.023811,157694.973482,57.089315,0,0


## 8. 전체 가구 자가점유비율

분모는 점유형태 1–7에 해당하고 조사 가중치가 양수인 전체 가구의 가중치 합, 분자는 그중 자가(1) 가구의 가중치 합이다. 전국은 시도 결과의 평균이 아니라 원자료 전체 가구에서 직접 산출한다. 2016년 충남·세종 결합표본은 기존 규칙을 유지해 충남으로 산출하고 세종은 구조적 결측으로 둔다.

In [9]:
own_output = pd.DataFrame(
    {"지역": OWN_REGION_ORDER, "세부지표": "전체 가구 자가점유비율"}
)
for year in YEARS:
    own_output[str(year)] = own_by_year[year].reindex(OWN_REGION_ORDER).to_numpy()
own_output.loc[:, [str(year) for year in YEARS]] = own_output[[str(year) for year in YEARS]].round(
    1
)
jeju_own_2023 = float(own_output.loc[own_output["지역"].eq("제주"), "2023"].iloc[0])
jeju_own_2024 = float(own_output.loc[own_output["지역"].eq("제주"), "2024"].iloc[0])
assert jeju_own_2023 == 57.1
print(
    f"2023년 제주 전체 가구 자가점유비율: 원 계산값 {jeju_own_2023_raw:.10f}%, "
    f"반올림 {jeju_own_2023:.1f}%"
)
print(own_output.to_string(index=False))

2023년 제주 전체 가구 자가점유비율: 원 계산값 57.0893148298%, 반올림 57.1%
지역         세부지표  2016  2017  2018  2019  2020  2021  2022  2023  2024
전국 전체 가구 자가점유비율  56.8  57.7  57.7  58.0  57.9  57.3  57.5  57.4  58.4
서울 전체 가구 자가점유비율  42.0  42.9  43.3  42.7  42.2  43.5  44.1  44.0  44.1
부산 전체 가구 자가점유비율  61.3  61.7  62.3  62.2  61.8  59.7  59.6  59.7  60.8
대구 전체 가구 자가점유비율  59.3  59.4  59.4  59.8  59.2  58.4  58.5  58.8  60.2
인천 전체 가구 자가점유비율  58.4  59.6  59.6  60.2  59.2  60.9  60.9  60.4  61.5
광주 전체 가구 자가점유비율  61.8  62.5  62.4  63.1  62.9  61.1  61.2  61.2  62.5
대전 전체 가구 자가점유비율  53.7  53.9  53.4  53.8  53.1  51.7  51.9  52.3  53.1
울산 전체 가구 자가점유비율  62.8  63.5  64.0  64.1  64.9  63.9  63.7  63.7  64.1
세종 전체 가구 자가점유비율   NaN  52.1  52.7  53.3  52.2  51.9  54.4  57.4  58.4
경기 전체 가구 자가점유비율  52.7  53.0  53.0  53.5  53.7  55.3  55.9  56.0  57.2
강원 전체 가구 자가점유비율  61.4  64.1  64.2  64.6  65.6  61.8  62.2  62.5  63.2
충북 전체 가구 자가점유비율  64.9  66.1  66.3  66.1  66.8  62.3  62.1  62.0  62.8
충남 전체 가구 자가점유비율  63.9  67.1  66.8  

## 9. 임차가구 연간 주거비 HCC

점유형태 2–6의 임차가구를 대상으로 `HCC = 보증금 × 적용 전월세전환율 / 100 + 월세 × 12`를 계산하고, 지역별 가구 HCC를 조사 가중치 없이 산술평균한다. 무상거주(7)는 제외한다. 전세(2)의 월세 공란과 보증금 없는 임차(4–6)의 보증금 공란만 구조적 0으로 처리하며, 필수 금액의 일반 결측·`9999999`·음수·비정상값은 유효 표본에서 제외한다.

원 보고서는 월별 전환율의 1–12월 평균 사용을 명시하지만 중간 정밀도 처리 규칙은 제시하지 않는다. 본계열은 월별 원 정밀도 값을 KOSIS 화면 표시 정밀도인 소수 첫째 자리로 사사오입한 뒤 평균하고, 그 연평균을 다시 소수 첫째 자리로 사사오입한다. 이는 제주 공식값 611만원을 정상적인 반올림으로 재현하는 가장 유력한 표시 정밀도 경로이며 공식 계산법으로 단정하지 않는다. 월별 원 정밀도 값의 단순평균을 적용한 결과는 민감도 계열로 함께 보존한다.

In [10]:
from decimal import Decimal, ROUND_HALF_UP

HCC_REGION_ORDER = OUTPUT_REGION_ORDER
HCC_EXPECTED_MISSING = EXPECTED_STRUCTURAL_MISSING
HCC_SOURCE_COLUMNS = ["시도", COL_TENURE, COL_DEPOSIT, COL_RENT]
HCC_RATE_QUANTUM = Decimal("0.1")
assert all("가중치" not in column for column in HCC_SOURCE_COLUMNS)


def half_up(value, quantum):
    return Decimal(str(value)).quantize(Decimal(quantum), rounding=ROUND_HALF_UP)


def hcc_rate_stats(region, year):
    monthly = (
        conversion_long.loc[
            conversion_long["지역"].eq(region) & conversion_long["연도"].eq(year),
            ["월", "전월세전환율"],
        ]
        .sort_values("월")
        .reset_index(drop=True)
    )
    raw_values = monthly["전월세전환율"].tolist()
    raw_decimals = [Decimal(str(value)) for value in raw_values]
    displayed_decimals = [
        value.quantize(HCC_RATE_QUANTUM, rounding=ROUND_HALF_UP) for value in raw_decimals
    ]
    raw_mean = sum(raw_decimals, Decimal(0)) / Decimal(len(raw_decimals))
    displayed_mean = sum(displayed_decimals, Decimal(0)) / Decimal(len(displayed_decimals))
    applied_rate = displayed_mean.quantize(HCC_RATE_QUANTUM, rounding=ROUND_HALF_UP)
    return {
        "월별 값 수": len(raw_values),
        "원 정밀도 연평균": float(raw_mean),
        "화면 정밀도 월값 평균": float(displayed_mean),
        "재현용 적용 전환율": float(applied_rate),
        "월별 자료 동일식 파생 여부": len(raw_values) == 12
        and all(np.isfinite(raw_values))
        and applied_rate
        == (
            sum(displayed_decimals, Decimal(0)) / Decimal(12)
        ).quantize(HCC_RATE_QUANTUM, rounding=ROUND_HALF_UP),
    }


def load_hcc_year(year, path):
    frame = pd.read_csv(path, encoding="cp949", usecols=HCC_SOURCE_COLUMNS).rename(
        columns={
            "시도": "시도코드",
            COL_TENURE: "점유형태",
            COL_DEPOSIT: "보증금_원자료",
            COL_RENT: "월세_원자료",
        }
    )
    frame["시도코드"] = pd.to_numeric(frame["시도코드"], errors="coerce")
    frame["지역"] = frame["시도코드"].map(SURVEY_REGION_MAP)
    if year == 2016:
        frame.loc[frame["시도코드"].isin([29, 34]), "지역"] = "충남"
    assert frame["지역"].notna().all(), f"{year}년 미매핑 시도코드 존재"
    frame["점유형태"] = pd.to_numeric(frame["점유형태"], errors="coerce")
    frame["보증금"] = pd.to_numeric(frame["보증금_원자료"], errors="coerce")
    frame["월세"] = pd.to_numeric(frame["월세_원자료"], errors="coerce")
    return frame


def calculate_hcc_year(year, path):
    frame = load_hcc_year(year, path)
    tenure = frame["점유형태"]
    deposit = frame["보증금"]
    rent = frame["월세"]
    renter_candidate = tenure.isin(range(2, 7))
    deposit_required = tenure.isin([2, 3])
    rent_required = tenure.isin([3, 4, 5, 6])
    structural_deposit_blank = tenure.isin([4, 5, 6]) & deposit.isna()
    structural_rent_blank = tenure.eq(2) & rent.isna()
    structural_unexpected = (
        (tenure.eq(2) & rent.notna() & rent.ne(0))
        | (tenure.isin([4, 5, 6]) & deposit.notna() & deposit.ne(0))
    )
    nonnumeric_amount = (
        (frame["보증금_원자료"].notna() & deposit.isna())
        | (frame["월세_원자료"].notna() & rent.isna())
    )
    explicit_missing = renter_candidate & (
        deposit.eq(INVALID_CODE) | rent.eq(INVALID_CODE)
    )
    nonfinite_amount = (
        (deposit.notna() & ~np.isfinite(deposit)) | (rent.notna() & ~np.isfinite(rent))
    )
    negative_amount = deposit.lt(0) | rent.lt(0)
    abnormal_amount = (
        renter_candidate
        & ~explicit_missing
        & (nonnumeric_amount | nonfinite_amount | negative_amount | structural_unexpected)
    )
    required_missing = renter_candidate & (
        (deposit_required & deposit.isna()) | (rent_required & rent.isna())
    )
    general_missing = required_missing & ~explicit_missing & ~abnormal_amount
    valid_hcc = renter_candidate & ~explicit_missing & ~abnormal_amount & ~general_missing

    frame["HCC용_보증금"] = deposit
    frame["HCC용_월세"] = rent
    frame.loc[structural_deposit_blank, "HCC용_보증금"] = 0
    frame.loc[structural_rent_blank, "HCC용_월세"] = 0
    assert frame.loc[valid_hcc, ["HCC용_보증금", "HCC용_월세"]].notna().all().all()
    assert (frame.loc[valid_hcc, ["HCC용_보증금", "HCC용_월세"]] >= 0).all().all()

    indicators = {}
    qa_rows = []
    for region in HCC_REGION_ORDER:
        region_mask = (
            pd.Series(True, index=frame.index)
            if region == "전국"
            else frame["지역"].eq(region)
        )
        rate = hcc_rate_stats(region, year)
        region_valid = region_mask & valid_hcc
        main_values = (
            frame["HCC용_보증금"] * rate["재현용 적용 전환율"] / 100
            + frame["HCC용_월세"] * 12
        )
        raw_values = (
            frame["HCC용_보증금"] * rate["원 정밀도 연평균"] / 100
            + frame["HCC용_월세"] * 12
        )
        main_hcc = float(main_values.loc[region_valid].mean()) if region_valid.any() else np.nan
        raw_hcc = float(raw_values.loc[region_valid].mean()) if region_valid.any() else np.nan
        indicators[region] = main_hcc
        qa_rows.append(
            {
                "연도": year,
                "지역": region,
                "산출 상태": (
                    "구조적 결측(충남·세종 결합표본)"
                    if (year, region) in HCC_EXPECTED_MISSING
                    else "산출"
                ),
                "전체 원표본 수": int(region_mask.sum()),
                **{
                    f"점유형태 {code} 표본 수": int((region_mask & tenure.eq(code)).sum())
                    for code in range(1, 8)
                },
                "점유형태 기타·결측 표본 수": int(
                    (region_mask & ~tenure.isin(range(1, 8))).sum()
                ),
                "임차 후보 표본 수": int((region_mask & renter_candidate).sum()),
                "유효 HCC 표본 수": int(region_valid.sum()),
                "구조적 월세 0 처리 수": int(
                    (region_mask & structural_rent_blank).sum()
                ),
                "구조적 보증금 0 처리 수": int(
                    (region_mask & structural_deposit_blank).sum()
                ),
                "9999999 제외 수": int((region_mask & explicit_missing).sum()),
                "일반 결측 제외 수": int((region_mask & general_missing).sum()),
                "음수·비정상값 제외 수": int((region_mask & abnormal_amount).sum()),
                "구조적 비해당항목 예상외 값 수": int(
                    (region_mask & renter_candidate & structural_unexpected).sum()
                ),
                "KOSIS 월별 값 수": rate["월별 값 수"],
                "KOSIS 월별 값 12개 존재 여부": rate["월별 값 수"] == 12,
                "원 정밀도 연평균": rate["원 정밀도 연평균"],
                "화면 정밀도 월값 평균": rate["화면 정밀도 월값 평균"],
                "재현용 적용 전환율": rate["재현용 적용 전환율"],
                "월별 자료 동일식 파생 여부": rate["월별 자료 동일식 파생 여부"],
                "평균 보증금": float(frame.loc[region_valid, "HCC용_보증금"].mean())
                if region_valid.any()
                else np.nan,
                "평균 월세": float(frame.loc[region_valid, "HCC용_월세"].mean())
                if region_valid.any()
                else np.nan,
                "본계열 HCC": main_hcc,
                "원 정밀도 민감도 HCC": raw_hcc,
                "두 HCC의 차이": main_hcc - raw_hcc
                if np.isfinite(main_hcc) and np.isfinite(raw_hcc)
                else np.nan,
            }
        )

    return pd.Series(indicators, dtype=float), pd.DataFrame(qa_rows)


hcc_by_year = {}
hcc_qa_frames = []
for year in YEARS:
    hcc_by_year[year], year_qa = calculate_hcc_year(year, housing_files[year])
    hcc_qa_frames.append(year_qa)

hcc_qa = pd.concat(hcc_qa_frames, ignore_index=True)
calculated_hcc_qa = hcc_qa.loc[hcc_qa["본계열 HCC"].notna()].copy()
observed_hcc_missing = set(
    hcc_qa.loc[hcc_qa["본계열 HCC"].isna(), ["연도", "지역"]].itertuples(
        index=False, name=None
    )
)
hcc_result_counts = calculated_hcc_qa.groupby("연도")["지역"].nunique()
jeju_hcc_2023 = hcc_qa.loc[
    hcc_qa["연도"].eq(2023) & hcc_qa["지역"].eq("제주")
].iloc[0]
jeju_hcc_2023_display = int(half_up(jeju_hcc_2023["본계열 HCC"], "1"))

finite_columns = [
    "원 정밀도 연평균",
    "화면 정밀도 월값 평균",
    "재현용 적용 전환율",
    "본계열 HCC",
    "원 정밀도 민감도 HCC",
]
hcc_qa_checks = pd.DataFrame(
    [
        {"검증 항목": "2016–2024년 누락 없음", "결과": "PASS" if set(hcc_by_year) == set(YEARS) else "FAIL"},
        {"검증 항목": "모든 지역·연도 KOSIS 월별 값 12개", "결과": "PASS" if hcc_qa["KOSIS 월별 값 수"].eq(12).all() else "FAIL"},
        {"검증 항목": "전환율·HCC 유한 비음수", "결과": "PASS" if (np.isfinite(calculated_hcc_qa[finite_columns]).all().all() and (calculated_hcc_qa[finite_columns] >= 0).all().all()) else "FAIL"},
        {"검증 항목": "유효 HCC 표본 수 ≤ 임차 후보", "결과": "PASS" if (hcc_qa["유효 HCC 표본 수"] <= hcc_qa["임차 후보 표본 수"]).all() else "FAIL"},
        {"검증 항목": "조사 가중치 미사용", "결과": "PASS" if all("가중치" not in column for column in HCC_SOURCE_COLUMNS) else "FAIL"},
        {"검증 항목": "예상외 지역 누락 없음", "결과": "PASS" if observed_hcc_missing == HCC_EXPECTED_MISSING and hcc_result_counts.loc[2016] == 17 and hcc_result_counts.drop(index=2016).eq(18).all() else "FAIL"},
        {"검증 항목": "2016년 세종 구조적 결측", "결과": "PASS" if observed_hcc_missing == {(2016, "세종")} else "FAIL"},
        {"검증 항목": "2023년 제주 임차 후보 570·유효 567", "결과": "PASS" if jeju_hcc_2023["임차 후보 표본 수"] == 570 and jeju_hcc_2023["유효 HCC 표본 수"] == 567 else "FAIL"},
        {"검증 항목": "2023년 제주 평균 보증금·월세", "결과": "PASS" if np.isclose(jeju_hcc_2023["평균 보증금"], 3138.4056437, atol=5e-8) and np.isclose(jeju_hcc_2023["평균 월세"], 34.9858907, atol=5e-8) else "FAIL"},
        {"검증 항목": "2023년 제주 적용 전환율 6.1%", "결과": "PASS" if jeju_hcc_2023["재현용 적용 전환율"] == 6.1 else "FAIL"},
        {"검증 항목": "2023년 제주 본계열 약 611.2734만원", "결과": "PASS" if np.isclose(jeju_hcc_2023["본계열 HCC"], 611.2734, atol=5e-5) else "FAIL"},
        {"검증 항목": "2023년 제주 정수 표시 611만원", "결과": "PASS" if jeju_hcc_2023_display == 611 else "FAIL"},
        {"검증 항목": "2023년 제주 원 정밀도 민감도 약 613.0697만원", "결과": "PASS" if np.isclose(jeju_hcc_2023["원 정밀도 민감도 HCC"], 613.0697, atol=5e-5) else "FAIL"},
        {"검증 항목": "월별 자료 동일식 파생(상수 대입 없음)", "결과": "PASS" if hcc_qa["월별 자료 동일식 파생 여부"].all() else "FAIL"},
    ]
)
assert hcc_qa_checks["결과"].eq("PASS").all(), hcc_qa_checks

hcc_output = pd.DataFrame(
    {"지역": HCC_REGION_ORDER, "세부지표": "임차가구 연간 주거비 HCC"}
)
for year in YEARS:
    hcc_output[str(year)] = hcc_by_year[year].reindex(HCC_REGION_ORDER).map(
        lambda value: float(half_up(value, "1")) if pd.notna(value) else np.nan
    ).to_numpy()

national_hcc_qa = hcc_qa.loc[hcc_qa["지역"].eq("전국")]
print(hcc_qa_checks.to_string(index=False))
print(
    f"HCC 결과: {len(calculated_hcc_qa)}/{len(hcc_qa)}개 지역·연도, "
    f"구조적 결측: {sorted(observed_hcc_missing)}"
)
print(
    "2023년 제주 — "
    f"후보 {int(jeju_hcc_2023['임차 후보 표본 수'])}가구, "
    f"유효 {int(jeju_hcc_2023['유효 HCC 표본 수'])}가구, "
    f"적용률 {jeju_hcc_2023['재현용 적용 전환율']:.1f}%, "
    f"본계열 {jeju_hcc_2023['본계열 HCC']:.10f}만원 → {jeju_hcc_2023_display}만원, "
    f"원 정밀도 {jeju_hcc_2023['원 정밀도 민감도 HCC']:.10f}만원"
)
print(hcc_output.to_string(index=False))
hcc_qa.loc[
    hcc_qa["연도"].eq(2023) & hcc_qa["지역"].eq("제주"),
]

                          검증 항목   결과
               2016–2024년 누락 없음 PASS
        모든 지역·연도 KOSIS 월별 값 12개 PASS
                 전환율·HCC 유한 비음수 PASS
            유효 HCC 표본 수 ≤ 임차 후보 PASS
                     조사 가중치 미사용 PASS
                   예상외 지역 누락 없음 PASS
                2016년 세종 구조적 결측 PASS
      2023년 제주 임차 후보 570·유효 567 PASS
             2023년 제주 평균 보증금·월세 PASS
           2023년 제주 적용 전환율 6.1% PASS
      2023년 제주 본계열 약 611.2734만원 PASS
           2023년 제주 정수 표시 611만원 PASS
2023년 제주 원 정밀도 민감도 약 613.0697만원 PASS
         월별 자료 동일식 파생(상수 대입 없음) PASS
HCC 결과: 161/162개 지역·연도, 구조적 결측: [(2016, '세종')]
2023년 제주 — 후보 570가구, 유효 567가구, 적용률 6.1%, 본계열 611.2734320988만원 → 611만원, 원 정밀도 613.0697121295만원
지역            세부지표  2016  2017  2018  2019  2020   2021   2022   2023   2024
전국 임차가구 연간 주거비 HCC 665.0 642.0 638.0 642.0 659.0  769.0  781.0  830.0  832.0
서울 임차가구 연간 주거비 HCC 897.0 931.0 912.0 927.0 916.0 1178.0 1082.0 1237.0 1196.0
부산 임차가구 연간 주거비 HCC 463.0 551.0 555.0 541.0 576.0  590.0  666.0  642.0  65

,연도,지역,산출 상태,전체 원표본 수,점유형태 1 표본 수,점유형태 2 표본 수,점유형태 3 표본 수,점유형태 4 표본 수,점유형태 5 표본 수,점유형태 6 표본 수,점유형태 7 표본 수,점유형태 기타·결측 표본 수,임차 후보 표본 수,유효 HCC 표본 수,구조적 월세 0 처리 수,...,9999999 제외 수,일반 결측 제외 수,음수·비정상값 제외 수,구조적 비해당항목 예상외 값 수,KOSIS 월별 값 수,KOSIS 월별 값 12개 존재 여부,원 정밀도 연평균,화면 정밀도 월값 평균,재현용 적용 전환율,월별 자료 동일식 파생 여부,평균 보증금,평균 월세,본계열 HCC,원 정밀도 민감도 HCC,두 HCC의 차이
143,2023,제주,산출,1824,1115,74,359,17,120,0,139,0,570,567,74,...,3,0,0,0,12,True,6.157235,6.141667,6.1,True,3138.405644,34.985891,611.273432,613.069712,-1.79628


## 10. 본계열 출력 구조와 구조적 결측

자가점유비율과 HCC 본계열은 전국과 17개 시도를 같은 순서로 배치한다. 2016년 세종은 충남·세종 결합표본 규칙에 따른 구조적 결측으로 유지하며 충남 값이나 다른 연도 값을 복사하지 않는다.

In [11]:
expected_columns = ["지역", "세부지표", *[str(year) for year in YEARS]]
for label, output in {"자가점유비율": own_output, "HCC": hcc_output}.items():
    assert output.shape == (18, 11), (label, output.shape)
    assert output.columns.tolist() == expected_columns
    assert output["지역"].tolist() == OUTPUT_REGION_ORDER
    assert pd.isna(output.loc[output["지역"].eq("세종"), "2016"].iloc[0])
print("자가점유비율·HCC: 전국 포함 18행, 공통 열·지역 순서, 2016년 세종 구조적 결측 확인")

자가점유비율·HCC: 전국 포함 18행, 공통 열·지역 순서, 2016년 세종 구조적 결측 확인


## 11. 현재 주택가격 산출

0717 파일에서 `현재 주택가격(만원)`의 유효 응답을 제주 원 보고서의 `1/n` 정의에 따라 전국·시도별 비가중 산술평균한다. 조사 가중치를 사용하지 않는다. `9999999`는 결측으로 제외하고 0·음수 부재를 검증한다. 값이 있는 응답은 원자료상 모두 자가(1)이지만 별도 점유형태 필터는 추가하지 않는다. 코드북이 없으므로 헤더 단위·결측 코드·2023년 제주 공식 기준값 재현을 판단 근거로 사용한 것이 한계다. 2016년 코드 29·34는 충남·세종 결합표본으로 충남에 산출하고 세종은 결측으로 둔다.

In [12]:
PRICE_SOURCE_COLUMNS = ["시도", COL_TENURE, COL_PRICE]
assert all("가중치" not in column for column in PRICE_SOURCE_COLUMNS)

price_by_year = {}
price_national_by_year = {}
price_qa_rows = []
for year in YEARS:
    frame = pd.read_csv(
        price_files[year], encoding="cp949", usecols=PRICE_SOURCE_COLUMNS
    )
    frame["시도코드"] = pd.to_numeric(frame["시도"], errors="coerce")
    frame["지역"] = frame["시도코드"].map(SURVEY_REGION_MAP)
    assert frame["지역"].notna().all(), f"0717 {year}년 미매핑 시도코드"
    if year == 2016:
        frame.loc[frame["시도코드"].isin([29, 34]), "지역"] = "충남"
    frame["점유형태"] = pd.to_numeric(frame[COL_TENURE], errors="coerce")
    frame["주택가격"] = pd.to_numeric(frame[COL_PRICE], errors="coerce")
    explicit_missing = frame["주택가격"].eq(INVALID_CODE)
    valid_price = frame["주택가격"].notna() & ~explicit_missing
    assert not (valid_price & frame["주택가격"].le(0)).any(), f"{year}년 주택가격 0·음수 존재"
    assert not (valid_price & frame["점유형태"].ne(1)).any(), f"{year}년 자가 외 주택가격 응답 존재"
    regional_price = (
        frame.loc[valid_price]
        .groupby("지역", observed=True)["주택가격"]
        .mean()
        .reindex(REGION_ORDER)
    )
    if year == 2016:
        regional_price.loc["세종"] = np.nan
    price_national_by_year[year] = float(frame.loc[valid_price, "주택가격"].mean())
    price_by_year[year] = pd.concat(
        [pd.Series({"전국": price_national_by_year[year]}), regional_price]
    ).reindex(OUTPUT_REGION_ORDER)
    price_qa_rows.append(
        {
            "연도": year,
            "전체 행": len(frame),
            "유효 주택가격": int(valid_price.sum()),
            "공란·비해당 제외": int(frame["주택가격"].isna().sum()),
            "9999999 제외": int(explicit_missing.sum()),
            "0·음수": int((frame["주택가격"].notna() & frame["주택가격"].le(0)).sum()),
            "자가 외 유효응답": int((valid_price & frame["점유형태"].ne(1)).sum()),
        }
    )

price_qa = pd.DataFrame(price_qa_rows)
price_output = pd.DataFrame({"지역": OUTPUT_REGION_ORDER, "세부지표": "현재 주택가격"})
for year in YEARS:
    price_output[str(year)] = price_by_year[year].reindex(OUTPUT_REGION_ORDER).to_numpy()
price_output.loc[:, [str(year) for year in YEARS]] = price_output[
    [str(year) for year in YEARS]
].round(1)
jeju_price_2023_raw = float(price_by_year[2023].loc["제주"])
jeju_2023_frame = pd.read_csv(price_files[2023], encoding="cp949", usecols=["시도", COL_PRICE])
jeju_2023_price = pd.to_numeric(jeju_2023_frame[COL_PRICE], errors="coerce")
jeju_2023_valid = (
    pd.to_numeric(jeju_2023_frame["시도"], errors="coerce").eq(39)
    & jeju_2023_price.notna()
    & jeju_2023_price.ne(INVALID_CODE)
)
assert int(jeju_2023_valid.sum()) == 1_111
assert np.isclose(jeju_price_2023_raw, 30_848.964896489648)
jeju_price_2023_display = int(half_up(jeju_price_2023_raw, "1"))
print(price_qa.to_string(index=False))
print(
    f"2023년 제주: 유효값 1,111건, 원값 {jeju_price_2023_raw:.12f}만원, 정수 표시 {jeju_price_2023_display:,}만원"
)
print(price_output.to_string(index=False))

  연도  전체 행  유효 주택가격  공란·비해당 제외  9999999 제외  0·음수  자가 외 유효응답
2016 20133    12140       7767         226     0          0
2017 60640    37447      22247         946     0          0
2018 61275    36658      23326        1291     0          0
2019 61170    36426      23993         751     0          0
2020 51421    29942      20332        1147     0          0
2021 51331    30452      19538        1341     0          0
2022 51325    30746      19486        1093     0          0
2023 61260    36669      23570        1021     0          0
2024 61341    36316      23656        1369     0          0
2023년 제주: 유효값 1,111건, 원값 30848.964896489648만원, 정수 표시 30,849만원
지역    세부지표    2016    2017    2018    2019    2020    2021    2022    2023    2024
전국 현재 주택가격 23866.7 22647.2 24621.1 25752.1 28565.0 35714.6 37470.4 33927.7 35030.5
서울 현재 주택가격 48213.9 53171.7 60586.9 64337.6 70379.2 89080.9 97210.4 91410.5 96805.8
부산 현재 주택가격 21813.4 25148.9 25802.9 25365.3 27416.2 34120.1 36832.5 34289.0 32543.5
대구 현재 

## 12. 중위매매가격 비교 보조값 구성

`(월) 중위매매가격_주택종합.csv`의 전국·17개 시도, 2016–2024년 108개월만 사용한다. 사용자 확인에 따라 원자료 값의 단위를 천원으로 보고 10으로 나눠 만원으로 변환한다. 각 지역·연도 12개월 완전성을 확인한 뒤 산술평균과 12월 값을 각각 만든다. 파일 안에는 단위 메타데이터가 없다는 점을 한계로 남긴다.

In [13]:
median_matches = sorted(RAW_ROOT.glob("(월) 중위매매가격_주택종합.csv"))
assert len(median_matches) == 1, f"중위매매가격 파일 후보 {len(median_matches)}개: {median_matches}"
MEDIAN_PATH = median_matches[0]
median_raw = pd.read_csv(MEDIAN_PATH, encoding="cp949", dtype=str)
median_month_pattern = re.compile(r"^(20(?:16|17|18|19|20|21|22|23|24))년 (\d{1,2})월$")
median_month_columns = [
    column for column in median_raw.columns if median_month_pattern.fullmatch(str(column))
]
assert len(median_month_columns) == 9 * 12
median_selected = median_raw.loc[
    median_raw["지역"].isin(["전국", *REGION_ORDER]), ["지역", *median_month_columns]
].copy()
assert len(median_selected) == 18 and median_selected["지역"].nunique() == 18
median_monthly = median_selected.melt(id_vars=["지역"], var_name="기준연월", value_name="원값_천원")
median_date_parts = median_monthly["기준연월"].str.extract(median_month_pattern)
median_monthly["연도"] = median_date_parts[0].astype(int)
median_monthly["월"] = median_date_parts[1].astype(int)
median_monthly["원값_천원"] = pd.to_numeric(
    median_monthly["원값_천원"].str.replace('"', "", regex=False).str.replace(",", "", regex=False),
    errors="coerce",
)
assert median_monthly["원값_천원"].notna().all()
median_monthly["중위매매가격_만원"] = median_monthly["원값_천원"] / 10
median_month_counts = median_monthly.groupby(["지역", "연도"], observed=True)["월"].nunique()
median_value_counts = median_monthly.groupby(["지역", "연도"], observed=True)[
    "중위매매가격_만원"
].count()
assert median_month_counts.size == 18 * 9
assert median_month_counts.eq(12).all() and median_value_counts.eq(12).all()
median_annual = median_monthly.groupby(["지역", "연도"], observed=True)["중위매매가격_만원"].mean()
median_december = median_monthly.loc[median_monthly["월"].eq(12)].set_index(["지역", "연도"])[
    "중위매매가격_만원"
]
unit_check_row = median_monthly.iloc[0]
assert np.isclose(unit_check_row["원값_천원"] / 10, unit_check_row["중위매매가격_만원"])
print(
    f"중위매매가격: 18개 지역 × 9개 연도 = {median_month_counts.size}개 지역-연도 모두 12개월 완전"
)
print("단위 변환 확인: 천원 원값 ÷ 10 = 만원")
print(median_annual.unstack("연도").round(1).to_string())

중위매매가격: 18개 지역 × 9개 연도 = 162개 지역-연도 모두 12개월 완전
단위 변환 확인: 천원 원값 ÷ 10 = 만원
연도     2016     2017     2018     2019     2020     2021     2022     2023     2024
지역                                                                                 
강원  11687.1  11887.2  13022.2  13096.8  13150.0  13166.1  13387.6  13517.7  13619.0
경기  25532.6  26094.8  28184.9  29768.3  33266.9  40203.0  43938.7  37937.5  38242.8
경남  15936.4  15897.6  15801.6  15650.6  15635.3  16063.0  16369.7  15488.2  15350.6
경북  11078.7  10928.7  10233.9  10311.6  10290.3  11064.3  11634.4  11563.6  11571.2
광주  15155.7  15182.7  17042.4  18630.8  18982.2  20328.8  21782.6  20440.7  20579.9
대구  21695.2  21934.2  23651.5  25157.2  26766.4  29803.5  29653.5  25869.6  25306.5
대전  18308.1  18727.1  19861.4  21937.4  25810.5  29862.7  29979.9  27203.2  27251.1
부산  19037.4  20019.2  22060.1  22027.0  23178.0  26323.7  27807.6  25033.5  24670.1
서울  43303.8  44924.6  54233.2  61181.2  65229.3  69929.2  72135.3  65938.0  67206.1
세종 

## 13. 현재 주택가격·비교 보조값 확인

두 계열이 모두 있는 지역·연도만 차이·차이율·상관·증감방향 비교에 포함한다. 순위는 연도별 내림차순이며 동률에는 최소 순위를 부여한다. 비교 결과는 노트북 안에만 남기고 기존 대체지표나 구조환경지표 파일을 수정하지 않는다.

In [14]:
comparison_rows = []
for year in YEARS:
    for region in REGION_ORDER:
        comparison_rows.append(
            {
                "지역": region,
                "연도": year,
                "주거실태조사_주택가격": price_by_year[year].get(region, np.nan),
                "중위매매가격_연평균": median_annual.get((region, year), np.nan),
                "중위매매가격_12월": median_december.get((region, year), np.nan),
            }
        )
comparison = pd.DataFrame(comparison_rows)
comparison["차이"] = comparison["주거실태조사_주택가격"] - comparison["중위매매가격_연평균"]
comparison["차이율"] = comparison["차이"] / comparison["중위매매가격_연평균"] * 100
comparison["원지표_순위"] = comparison.groupby("연도")["주거실태조사_주택가격"].rank(
    method="min", ascending=False
)
comparison["대체지표_순위"] = comparison.groupby("연도")["중위매매가격_연평균"].rank(
    method="min", ascending=False
)
comparison["순위차"] = comparison["원지표_순위"] - comparison["대체지표_순위"]
comparison["연평균-12월"] = comparison["중위매매가격_연평균"] - comparison["중위매매가격_12월"]
comparison["원지표_전년차"] = comparison.groupby("지역", sort=False)["주거실태조사_주택가격"].diff()
comparison["대체지표_전년차"] = comparison.groupby("지역", sort=False)["중위매매가격_연평균"].diff()
direction_valid = comparison["원지표_전년차"].notna() & comparison["대체지표_전년차"].notna()
comparison["증감방향_일치"] = pd.Series(pd.NA, index=comparison.index, dtype="boolean")
comparison.loc[direction_valid, "증감방향_일치"] = np.sign(
    comparison.loc[direction_valid, "원지표_전년차"]
).eq(np.sign(comparison.loc[direction_valid, "대체지표_전년차"]))

yearly_corr_rows = []
for year, group in comparison.groupby("연도", sort=True):
    pairs = group[["주거실태조사_주택가격", "중위매매가격_연평균"]].dropna()
    yearly_corr_rows.append(
        {
            "연도": year,
            "표본수": len(pairs),
            "Pearson": pairs.iloc[:, 0].corr(pairs.iloc[:, 1], method="pearson"),
            "Spearman": pairs.iloc[:, 0].corr(pairs.iloc[:, 1], method="spearman"),
        }
    )
yearly_correlations = pd.DataFrame(yearly_corr_rows)

regional_corr_rows = []
for region, group in comparison.groupby("지역", sort=False):
    pairs = group[["주거실태조사_주택가격", "중위매매가격_연평균"]].dropna()
    regional_corr_rows.append(
        {
            "지역": region,
            "표본수": len(pairs),
            "Pearson": pairs.iloc[:, 0].corr(pairs.iloc[:, 1], method="pearson"),
            "Spearman": pairs.iloc[:, 0].corr(pairs.iloc[:, 1], method="spearman"),
        }
    )
regional_correlations = pd.DataFrame(regional_corr_rows)
direction_summary = (
    comparison.groupby("연도", sort=True)["증감방향_일치"]
    .agg(표본수="count", 일치율=lambda values: values.mean() * 100)
    .reset_index()
)
annual_december_summary = (
    comparison.groupby("연도", sort=True)["연평균-12월"]
    .agg(["count", "mean", "min", "max"])
    .reset_index()
)

print("[2023년 제주 비교 보조값]")
print(
    comparison.loc[
        comparison["지역"].eq("제주") & comparison["연도"].eq(2023)
    ].round(2).to_string(index=False)
)
print("\n[연도별 시도 횡단면 상관]")
print(yearly_correlations.round(4).to_string(index=False))
print("\n[지역별 2016–2024 시계열 상관]")
print(regional_correlations.round(4).to_string(index=False))
print("\n[전년 대비 증감방향 일치]")
print(direction_summary.round(2).to_string(index=False))
print("\n[중위매매가격 연평균-12월 차이]")
print(annual_december_summary.round(2).to_string(index=False))

[2023년 제주 비교 보조값]
지역   연도  주거실태조사_주택가격  중위매매가격_연평균  중위매매가격_12월      차이   차이율  원지표_순위  대체지표_순위  순위차  연평균-12월  원지표_전년차  대체지표_전년차  증감방향_일치
제주 2023     30848.96    22855.52     22785.5 7993.44 34.97     9.0      9.0  0.0    70.02 -4167.01   -711.72     True

[연도별 시도 횡단면 상관]
  연도  표본수  Pearson  Spearman
2016   16   0.9867    0.9412
2017   17   0.9847    0.9363
2018   17   0.9906    0.9828
2019   17   0.9918    0.9681
2020   17   0.9935    0.9755
2021   17   0.9790    0.9877
2022   17   0.9509    0.9681
2023   17   0.9586    0.9681
2024   17   0.9669    0.9730

[지역별 2016–2024 시계열 상관]
지역  표본수  Pearson  Spearman
서울    9   0.9105    0.9500
부산    9   0.9659    0.9833
대구    9   0.9469    0.9167
인천    9   0.9626    0.9833
광주    9   0.9836    0.9667
대전    9   0.9797    0.9667
울산    9   0.9341    0.8167
세종    8   0.9821    0.9524
경기    9   0.9772    0.9833
강원    9   0.7723    0.9333
충북    9   0.9555    0.8500
충남    9   0.9323    0.9833
전북    9   0.3955    0.0000
전남    9   0.8189    0.5000
경북    9   

## 14. 최종 세 지표 핵심값과 교차 QA

세 본계열의 산식·가중치 정책·전국 포함 18행 구조·2016년 세종 구조적 결측을 함께 검증한다. HCC 원 정밀도 결과는 민감도 QA에만 존재하고 본계열 결과표에는 포함하지 않는다.

In [15]:
output_frames = {
    "전체 가구 자가점유비율": own_output,
    "임차가구 연간 주거비 HCC": hcc_output,
    "현재 주택가격": price_output,
}
expected_indicator_names = set(output_frames)
actual_indicator_names = {frame["세부지표"].iloc[0] for frame in output_frames.values()}
all_output_structure_ok = all(
    frame.shape == (18, 11)
    and frame.columns.tolist() == expected_columns
    and frame["지역"].tolist() == OUTPUT_REGION_ORDER
    for frame in output_frames.values()
)
all_sejong_missing = all(
    pd.isna(frame.loc[frame["지역"].eq("세종"), "2016"].iloc[0])
    for frame in output_frames.values()
)
ownership_weight_formula_ok = np.allclose(
    calculated_own_qa["자가점유비율"],
    calculated_own_qa["분자 가중치 합"] / calculated_own_qa["분모 가중치 합"] * 100,
)
hcc_sensitivity_separate = (
    "원 정밀도 민감도 HCC" in hcc_qa.columns
    and all("민감도" not in column and "원 정밀도" not in column for column in hcc_output.columns)
)

final_qa_checks = pd.DataFrame(
    [
        {"검증 항목": "최종 지표가 정확히 3종", "결과": "PASS" if actual_indicator_names == expected_indicator_names else "FAIL"},
        {"검증 항목": "세 결과표 전국 포함 18행·공통 순서", "결과": "PASS" if all_output_structure_ok else "FAIL"},
        {"검증 항목": "2016년 세종 세 지표 구조적 결측", "결과": "PASS" if all_sejong_missing else "FAIL"},
        {"검증 항목": "2023년 제주 자가점유비율 원값·57.1%", "결과": "PASS" if np.isclose(jeju_own_2023_raw, 57.0893148, atol=5e-8) and jeju_own_2023 == 57.1 else "FAIL"},
        {"검증 항목": "2023년 제주 HCC 유효 표본 567", "결과": "PASS" if jeju_hcc_2023["유효 HCC 표본 수"] == 567 else "FAIL"},
        {"검증 항목": "2023년 제주 HCC 적용률 6.1%", "결과": "PASS" if jeju_hcc_2023["재현용 적용 전환율"] == 6.1 else "FAIL"},
        {"검증 항목": "2023년 제주 HCC 본계열 약 611.2734만원", "결과": "PASS" if np.isclose(jeju_hcc_2023["본계열 HCC"], 611.2734, atol=5e-5) else "FAIL"},
        {"검증 항목": "2023년 제주 HCC 정수 표시 611만원", "결과": "PASS" if jeju_hcc_2023_display == 611 else "FAIL"},
        {"검증 항목": "2023년 제주 HCC 원 정밀도 약 613.0697만원", "결과": "PASS" if np.isclose(jeju_hcc_2023["원 정밀도 민감도 HCC"], 613.0697, atol=5e-5) else "FAIL"},
        {"검증 항목": "2023년 제주 현재 주택가격 약 30,848.9649만원", "결과": "PASS" if np.isclose(jeju_price_2023_raw, 30_848.9649, atol=5e-5) else "FAIL"},
        {"검증 항목": "2023년 제주 현재 주택가격 정수 표시 30,849만원", "결과": "PASS" if jeju_price_2023_display == 30_849 else "FAIL"},
        {"검증 항목": "자가점유비율 조사 가중치 사용", "결과": "PASS" if ownership_weight_formula_ok else "FAIL"},
        {"검증 항목": "HCC 조사 가중치 미사용", "결과": "PASS" if all("가중치" not in column for column in HCC_SOURCE_COLUMNS) else "FAIL"},
        {"검증 항목": "현재 주택가격 조사 가중치 미사용", "결과": "PASS" if all("가중치" not in column for column in PRICE_SOURCE_COLUMNS) else "FAIL"},
        {"검증 항목": "HCC 본계열·원 정밀도 민감도 분리", "결과": "PASS" if hcc_sensitivity_separate else "FAIL"},
        {"검증 항목": "지표별 자체 QA 통과", "결과": "PASS" if own_qa_checks["결과"].eq("PASS").all() and hcc_qa_checks["결과"].eq("PASS").all() else "FAIL"},
    ]
)
assert final_qa_checks["결과"].eq("PASS").all(), final_qa_checks
print(final_qa_checks.to_string(index=False))
pd.DataFrame(
    [
        {"지표": "전체 가구 자가점유비율", "2023년 제주 원값": jeju_own_2023_raw, "표시값": jeju_own_2023},
        {"지표": "임차가구 연간 주거비 HCC", "2023년 제주 원값": jeju_hcc_2023["본계열 HCC"], "표시값": jeju_hcc_2023_display},
        {"지표": "현재 주택가격", "2023년 제주 원값": jeju_price_2023_raw, "표시값": jeju_price_2023_display},
    ]
)

                           검증 항목   결과
                   최종 지표가 정확히 3종 PASS
           세 결과표 전국 포함 18행·공통 순서 PASS
            2016년 세종 세 지표 구조적 결측 PASS
        2023년 제주 자가점유비율 원값·57.1% PASS
          2023년 제주 HCC 유효 표본 567 PASS
           2023년 제주 HCC 적용률 6.1% PASS
   2023년 제주 HCC 본계열 약 611.2734만원 PASS
        2023년 제주 HCC 정수 표시 611만원 PASS
 2023년 제주 HCC 원 정밀도 약 613.0697만원 PASS
2023년 제주 현재 주택가격 약 30,848.9649만원 PASS
 2023년 제주 현재 주택가격 정수 표시 30,849만원 PASS
                자가점유비율 조사 가중치 사용 PASS
                  HCC 조사 가중치 미사용 PASS
              현재 주택가격 조사 가중치 미사용 PASS
            HCC 본계열·원 정밀도 민감도 분리 PASS
                    지표별 자체 QA 통과 PASS


,지표,2023년 제주 원값,표시값
0,전체 가구 자가점유비율,57.089315,57.1
1,임차가구 연간 주거비 HCC,611.273432,611.0
2,현재 주택가격,30848.964896,30849.0


## 15. 최종 본계열 CSV 저장

세 본계열을 각각 `지역 | 세부지표 | 2016 | … | 2024` 구조의 18행으로 UTF-8-SIG 저장한다. 파일명에는 현재 지표 정의를 사용한다. HCC 원 정밀도 민감도는 저장 대상에 포함하지 않는다.

In [16]:
OWN_PATH = PROCESSED_DIR / "전체_가구_자가점유비율_2016-2024.csv"
HCC_PATH = PROCESSED_DIR / "임차가구_연간_주거비_HCC_2016-2024.csv"
PRICE_PATH = PROCESSED_DIR / "현재_주택가격_2016-2024.csv"
csv_specs = {
    "전체 가구 자가점유비율": (OWN_PATH, own_output, "%.1f"),
    "임차가구 연간 주거비 HCC": (HCC_PATH, hcc_output, "%.0f"),
    "현재 주택가격": (PRICE_PATH, price_output, "%.1f"),
}
for label, (path, frame, float_format) in csv_specs.items():
    frame.to_csv(path, index=False, encoding="utf-8-sig", float_format=float_format)
    print(f"저장: {label} — {path} / shape={frame.shape}")

저장: 전체 가구 자가점유비율 — D:\University\yumocha\yumocha-issue49\data\processed\전체_가구_자가점유비율_2016-2024.csv / shape=(18, 11)
저장: 임차가구 연간 주거비 HCC — D:\University\yumocha\yumocha-issue49\data\processed\임차가구_연간_주거비_HCC_2016-2024.csv / shape=(18, 11)
저장: 현재 주택가격 — D:\University\yumocha\yumocha-issue49\data\processed\현재_주택가격_2016-2024.csv / shape=(18, 11)


## 16. CSV 재읽기 검증

저장된 세 CSV를 다시 읽어 인메모리 결과와의 일치, 18×11 구조, 지역·연도 순서, 숫자형 자료형, 불필요한 인덱스 열 부재, 2016년 세종 구조적 결측 및 HCC 민감도 비혼합을 검증한다.

In [17]:
reloaded = {
    label: pd.read_csv(path, encoding="utf-8-sig")
    for label, (path, _, _) in csv_specs.items()
}
csv_validation_rows = []
for label, data in reloaded.items():
    path, expected, _ = csv_specs[label]
    numeric = data[[str(year) for year in YEARS]]
    missing_locations = {
        (data.loc[row_index, "지역"], str(year))
        for row_index in data.index
        for year in YEARS
        if pd.isna(data.loc[row_index, str(year)])
    }
    checks = {
        "UTF-8 BOM": path.read_bytes().startswith(b"\xef\xbb\xbf"),
        "18×11 크기": data.shape == (18, 11),
        "열 순서": data.columns.tolist() == expected_columns,
        "지역 순서": data["지역"].tolist() == OUTPUT_REGION_ORDER,
        "숫자형 연도 열": all(pd.api.types.is_numeric_dtype(numeric[column]) for column in numeric),
        "불필요한 인덱스 열 없음": not any(column.startswith("Unnamed:") for column in data.columns),
        "구조적 결측 위치": missing_locations == {("세종", "2016")},
        "인메모리 결과 일치": True,
        "민감도 비혼합": label != "임차가구 연간 주거비 HCC" or (
            data["세부지표"].eq("임차가구 연간 주거비 HCC").all()
            and all("민감도" not in column and "원 정밀도" not in column for column in data.columns)
        ),
    }
    try:
        pd.testing.assert_frame_equal(data, expected, check_dtype=False, rtol=0, atol=0)
    except AssertionError:
        checks["인메모리 결과 일치"] = False
    for item, passed in checks.items():
        csv_validation_rows.append(
            {"CSV": label, "검증 항목": item, "결과": "PASS" if passed else "FAIL"}
        )

csv_validation_checks = pd.DataFrame(csv_validation_rows)
assert csv_validation_checks["결과"].eq("PASS").all(), csv_validation_checks
print(csv_validation_checks.to_string(index=False))
print("CSV 3개: 인메모리 일치·18×11·순서·자료형·인덱스·구조적 결측·민감도 분리 검증 완료")

            CSV         검증 항목   결과
   전체 가구 자가점유비율     UTF-8 BOM PASS
   전체 가구 자가점유비율      18×11 크기 PASS
   전체 가구 자가점유비율          열 순서 PASS
   전체 가구 자가점유비율         지역 순서 PASS
   전체 가구 자가점유비율      숫자형 연도 열 PASS
   전체 가구 자가점유비율 불필요한 인덱스 열 없음 PASS
   전체 가구 자가점유비율     구조적 결측 위치 PASS
   전체 가구 자가점유비율    인메모리 결과 일치 PASS
   전체 가구 자가점유비율       민감도 비혼합 PASS
임차가구 연간 주거비 HCC     UTF-8 BOM PASS
임차가구 연간 주거비 HCC      18×11 크기 PASS
임차가구 연간 주거비 HCC          열 순서 PASS
임차가구 연간 주거비 HCC         지역 순서 PASS
임차가구 연간 주거비 HCC      숫자형 연도 열 PASS
임차가구 연간 주거비 HCC 불필요한 인덱스 열 없음 PASS
임차가구 연간 주거비 HCC     구조적 결측 위치 PASS
임차가구 연간 주거비 HCC    인메모리 결과 일치 PASS
임차가구 연간 주거비 HCC       민감도 비혼합 PASS
        현재 주택가격     UTF-8 BOM PASS
        현재 주택가격      18×11 크기 PASS
        현재 주택가격          열 순서 PASS
        현재 주택가격         지역 순서 PASS
        현재 주택가격      숫자형 연도 열 PASS
        현재 주택가격 불필요한 인덱스 열 없음 PASS
        현재 주택가격     구조적 결측 위치 PASS
        현재 주택가격    인메모리 결과 일치 PASS
        현재 주택가격       민감도 비혼합 PASS
CSV 3개: 인메모리 일치·18×1